In [1]:
import pandas as pd
import numpy as np
# 读取数据
df = pd.read_excel('df4.xlsx') 
df.head()

: 

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
import re

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   605 non-null    int64  
 1   孕妇代码         605 non-null    int64  
 2   年龄           605 non-null    int64  
 3   身高           605 non-null    float64
 4   体重           605 non-null    float64
 5   检测抽血次数       605 non-null    int64  
 6   检测孕周         605 non-null    float64
 7   孕妇BMI        605 non-null    float64
 8   唯一比对的读段数     605 non-null    int64  
 9   GC含量         605 non-null    float64
 10  13号染色体的Z值    605 non-null    float64
 11  18号染色体的Z值    605 non-null    float64
 12  21号染色体的Z值    605 non-null    float64
 13  X染色体的Z值      605 non-null    float64
 14  Unnamed: 20  0 non-null      float64
 15  Unnamed: 21  0 non-null      float64
 16  X染色体浓度       605 non-null    float64
 17  13号染色体的GC含量  605 non-null    float64
 18  18号染色体的GC含量  605 non-null    float64
 19  21号染色体的G

In [ ]:

df.head()

,Unnamed: 0,孕妇代码,年龄,身高,体重,检测抽血次数,检测孕周,孕妇BMI,唯一比对的读段数,GC含量,...,21号染色体的GC含量,被过滤掉读段数的比例,染色体的非整倍体,生产次数,胎儿是否健康,T13,T18,T21,状态,怀孕次数_数值
0,0,1,32,162.0,82.0,1,13.714286,31.245237,4993952,0.402535,...,0.402970,0.024708,NaN,0,1,0,0,0,0,1
1,1,1,32,162.0,82.0,2,17.142857,31.245237,3997118,0.400171,...,0.399509,0.025608,NaN,0,1,0,0,0,0,1
2,2,1,32,162.0,85.0,3,19.857143,32.388355,4277620,0.407386,...,0.408611,0.024452,NaN,0,1,0,0,0,0,1
3,3,1,32,162.0,86.0,4,23.000000,32.769395,3617078,0.398090,...,0.397539,0.022412,NaN,0,1,0,0,0,0,1
4,4,2,33,165.0,96.0,1,12.714286,35.261708,3862611,0.398646,...,0.401632,0.022843,NaN,0,1,0,0,0,0,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 29 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   605 non-null    int64  
 1   孕妇代码         605 non-null    int64  
 2   年龄           605 non-null    int64  
 3   身高           605 non-null    float64
 4   体重           605 non-null    float64
 5   检测抽血次数       605 non-null    int64  
 6   检测孕周         605 non-null    float64
 7   孕妇BMI        605 non-null    float64
 8   唯一比对的读段数     605 non-null    int64  
 9   GC含量         605 non-null    float64
 10  13号染色体的Z值    605 non-null    float64
 11  18号染色体的Z值    605 non-null    float64
 12  21号染色体的Z值    605 non-null    float64
 13  X染色体的Z值      605 non-null    float64
 14  Unnamed: 20  0 non-null      float64
 15  Unnamed: 21  0 non-null      float64
 16  X染色体浓度       605 non-null    float64
 17  13号染色体的GC含量  605 non-null    float64
 18  18号染色体的GC含量  605 non-null    float64
 19  21号染色体的G

In [ ]:
# 预处理：将特征与标签全部数值化，消除 'T18' 等字符串导致的转换错误
raw_df = df.copy()

# 1. 构建标签 y（保持原逻辑：倒数第3列），若为字符串则编码
y = raw_df.iloc[:, -3].copy()
if y.dtype == 'object':
    y_codes, y_uniques = pd.factorize(y.astype(str).str.strip())
    print('标签列为 object，已编码映射(前10个):')
    print({i: v for i, v in enumerate(y_uniques[:10])})
    y = pd.Series(y_codes, index=y.index, name=y.name)
y = pd.to_numeric(y, errors='coerce').fillna(0).astype(int)

# 2. 原逻辑特征切片 -> 复制
X_raw = raw_df.iloc[:, 1:-4].copy()

# 3. 因子编码所有 object 列
obj_cols = [c for c in X_raw.columns if X_raw[c].dtype == 'object']
encoded_examples = {}
for c in obj_cols:
    codes, uniques = pd.factorize(X_raw[c].astype(str).str.strip())
    X_raw[c] = codes
    encoded_examples[c] = list(uniques[:8])  # 仅展示前8个原值

# 4. 缺失值填充
if X_raw.isna().any().any():
    miss_cols = X_raw.columns[X_raw.isna().any()].tolist()
    print('存在缺失列，使用0填充:', miss_cols)
    X_raw = X_raw.fillna(0)

# 5. 校验是否全部为数值
non_numeric = [c for c in X_raw.columns if not np.issubdtype(X_raw[c].dtype, np.number)]
if non_numeric:
    raise RuntimeError(f'仍存在非数值列: {non_numeric}')

# 6. 最终赋值
X = X_raw
print(f'X 形状: {X.shape} | y 形状: {y.shape} | y(均值≈正类占比): {y.mean():.4f}')
if encoded_examples:
    print('\n对象列编码示例 (列: 原值前若干 → 编码顺序即 0,1,...):')
    for k,v in encoded_examples.items():
        print(f'  {k}: {v}')
print('\n预处理完成，可继续划分训练集。')

存在缺失列，使用0填充: ['Unnamed: 20', 'Unnamed: 21']
X 形状: (605, 24) | y 形状: (605,) | y(均值≈正类占比): 0.0215

对象列编码示例 (列: 原值前若干 → 编码顺序即 0,1,...):
  染色体的非整倍体: ['nan', 'T21', 'T13', 'T18', 'T13T18', 'T18T21', 'T13T21']

预处理完成，可继续划分训练集。


In [ ]:
##孕妇的BMI第186行有一个缺失值，后来自行计算并添加了
nan_counts = X.isnull().sum()
print("各列的NaN数量:")
print(nan_counts[nan_counts > 0])  # 只显示有NaN的列
nan_indices = np.where(X['孕妇BMI'].isna())[0].tolist()
print(f"列 '孕妇BMI' 中 NaN 所在的行索引: {nan_indices}")

各列的NaN数量:
Series([], dtype: int64)
列 '孕妇BMI' 中 NaN 所在的行索引: []


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
import xgboost  #需要安装的库
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y, test_size=0.2, stratify=y,random_state=10)

In [ ]:
# 堆叠stacking算法
# 创建第一层的基本训练器列表
first_estimators=[("rf", RandomForestClassifier(random_state=0)),("ridge", make_pipeline(StandardScaler(),RidgeClassifier(random_state=0))),("svc", make_pipeline(StandardScaler(),SVC(random_state=0)))]
# 创建堆叠学习器分类模型
clf=StackingClassifier(estimators=first_estimators,final_estimator=LogisticRegression(max_iter=5000,random_state=0),passthrough=True)
# 训练模型
clf.fit(X_train,y_train)

,estimators,"[('rf', ...), ('ridge', ...), ...]"
,final_estimator,LogisticRegre...andom_state=0)
,cv,None
,stack_method,'auto'
,n_jobs,None
,passthrough,True
,verbose,0
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2


In [ ]:
print("训练集预测准确率：",clf.score(X_train,y_train),sep="")
print("测试集预测准确率：",clf.score(X_test,y_test),sep="")
print("测试集前3个样本的预测标签：",clf.predict(X_test[:3]))
print("测试集前3个样本的真实标签：",y_test[:3])
print("测试集前3个样本的预测标签概率：\n",
      clf.predict_proba(X_test[:3]),sep="")


训练集预测准确率：0.9834710743801653
测试集预测准确率：0.9752066115702479
测试集前3个样本的预测标签： [0 0 0]
测试集前3个样本的真实标签： 141    0
370    0
273    0
Name: T21, dtype: int64
测试集前3个样本的预测标签概率：
[[0.99703181 0.00296819]
 [0.9978111  0.0021889 ]
 [0.99836317 0.00163683]]


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
gbc = GradientBoostingClassifier(n_estimators=500)
gbc.fit(X_train,y_train)
print("训练集准确率：", gbc.score(X_train, y_train), sep="")
print("测试集准确率：", gbc.score(X_test, y_test), sep="") 

训练集准确率：1.0
测试集准确率：1.0


In [ ]:
y_pred = gbc.predict(X_test)
# 计算混淆矩阵
cm = confusion_matrix(y_test, y_pred)
print("混淆矩阵:")
print(cm)

混淆矩阵:
[[118   0]
 [  0   3]]


In [ ]:
dtc =  DecisionTreeClassifier()
rfc = RandomForestClassifier()
xgb = xgboost.XGBClassifier()

In [ ]:
clf = [dtc,rfc,xgb]
for algo in clf:
    score = cross_val_score( algo,X,y,cv = 5,scoring = 'accuracy')
    print("The accuracy score of {} is:".format(algo),score.mean())


The accuracy score of DecisionTreeClassifier() is: 0.8413223140495868
The accuracy score of RandomForestClassifier() is: 0.9702479338842975
The accuracy score of RandomForestClassifier() is: 0.9702479338842975
The accuracy score of XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...) is: 0.9

In [ ]:
clf1 = [('dtc',dtc),('rfc',rfc),('xgb',xgb)] #list of (str, estimator)

In [ ]:
lr = LogisticRegression()
stack_model = StackingClassifier( estimators = clf1,final_estimator = lr)
score = cross_val_score(stack_model,X,y,cv = 5,scoring = 'accuracy')
print("The accuracy score of is:",score.mean())

The accuracy score of is: 0.9785123966942149


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import lightgbm as lgb  #需要另外安装

# 标准化特征
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 转回 DataFrame 以保持列名，防止 LightGBM 特征名告警
feature_cols = X_train.columns.tolist()
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled_df  = pd.DataFrame(X_test_scaled,  columns=feature_cols, index=X_test.index)
print('特征列数量:', len(feature_cols), '| 示例:', feature_cols[:8])

model = lgb.LGBMClassifier(
    n_estimators=120,
    learning_rate=0.03,
    max_depth=-1,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)

model.fit(X_train_scaled_df, y_train)

y_pred = model.predict(X_test_scaled_df)
y_proba = model.predict_proba(X_test_scaled_df) if hasattr(model,'predict_proba') else None

accuracy = accuracy_score(y_test, y_pred)
print(f'模型准确率: {accuracy:.4f}')
cm = confusion_matrix(y_test, y_pred)
print('\n混淆矩阵:')
print(cm)
print('\n分类报告:')
print(classification_report(y_test, y_pred, zero_division=0))
if y_proba is not None:
    print('预测概率(前3行):')
    print(y_proba[:3])

# 特征重要性(按增益)展示前15个
importances = model.booster_.feature_importance(importance_type='gain')
fi = (pd.DataFrame({'feature': feature_cols, 'importance': importances})
        .sort_values('importance', ascending=False)
        .head(15))
print('\nTop15 特征重要性:')
print(fi.to_string(index=False))

[LightGBM] [Info] Number of positive: 10, number of negative: 474
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000317 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2249
[LightGBM] [Info] Number of data points in the train set: 484, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.020661 -> initscore=-3.858622
[LightGBM] [Info] Start training from score -3.858622
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

c:\Users\dell\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
cm

array([[118,   0],
       [  3,   0]])

In [ ]:
y_pred